# Experiment 8: Clustering Human Activity Recognition Data using K-Means, DBSCAN, and Hierarchical Clustering

**Course**: ICS1512 - Machine Learning Algorithms Laboratory
**Institution**: Sri Sivasubramaniya Nadar College of Engineering, Chennai
**Degree & Branch**: M.Tech (Integrated) Computer Science and Engineering, Semester V
**Student Name**: Arivuchezhiyan E
**Register Number**: 3122247001006
**Faculty**: Dr. Poreddy Ajay Kumar Reddy
**Date**: 20/09/2026

---

## 1. Objective
To implement and analyze the performance of unsupervised clustering algorithms on the Human Activity Recognition (HAR) dataset:
- Model A: K-Means Clustering (Elbow Method and Silhouette Analysis)
- Model B: DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
- Model C: Hierarchical Agglomerative Clustering (HAC with Single, Complete, Average, and Ward Linkage)


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score,
    homogeneity_completeness_v_measure, confusion_matrix
)
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
from scipy.spatial.distance import pdist
from scipy.optimize import linear_sum_assignment

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

SEED = 42
np.random.seed(SEED)
print('Libraries imported successfully.')


## 2. Dataset Loading and Preprocessing
The dataset contains 10,299 instances across 561 continuous sensor features captured at 50 Hz from a 3-axis accelerometer and gyroscope.


In [ ]:
with open('features.txt', 'r') as f:
    raw_features = [line.strip().split(' ', 1)[1] for line in f.readlines() if line.strip()]

seen = {}
features = []
for i, feat in enumerate(raw_features):
    if feat in seen:
        seen[feat] += 1
        features.append(feat + '_' + str(seen[feat]))
    else:
        seen[feat] = 0
        features.append(feat)

activities = {1: 'WALKING', 2: 'WALKING_UPSTAIRS', 3: 'WALKING_DOWNSTAIRS', 4: 'SITTING', 5: 'STANDING', 6: 'LAYING'}

X_train = pd.read_csv('X_train.txt', sep=r'\s+', header=None, names=features)
y_train = pd.read_csv('y_train.txt', header=None, names=['Activity_ID'])

X_test = pd.read_csv('X_test.txt', sep=r'\s+', header=None, names=features)
y_test = pd.read_csv('y_test.txt', header=None, names=['Activity_ID'])

X = pd.concat([X_train, X_test], ignore_index=True)
y_df = pd.concat([y_train, y_test], ignore_index=True)
y_df['Activity'] = y_df['Activity_ID'].map(activities)

print('Loaded samples:', X.shape[0], 'features:', X.shape[1])
print('Missing values:', X.isnull().sum().sum())


In [ ]:
le = LabelEncoder()
y_true = le.fit_transform(y_df['Activity'])
target_names = le.classes_

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA(random_state=SEED)
pca_full.fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.argmax(cum_var >= 0.95) + 1
print('Number of principal components for >= 95% variance:', n_95)

pca_2d = PCA(n_components=2, random_state=SEED)
X_pca2d = pca_2d.fit_transform(X_scaled)

pca_50 = PCA(n_components=50, random_state=SEED)
X_pca50 = pca_50.fit_transform(X_scaled)


## 3. Model A: K-Means Clustering and Elbow Method
We systematically vary k from 2 to 8 and compute WCSS (Inertia) and Silhouette Scores.


In [ ]:
k_values = list(range(2, 9))
wcss_vals = []
sil_vals = []
km_dict = {}

for k in k_values:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    wcss_vals.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=3000, random_state=SEED)
    sil_vals.append(sil)
    km_dict[k] = (km, labels)
    print('k =', k, '| WCSS =', round(km.inertia_, 2), '| Silhouette =', round(sil, 4))

table1_df = pd.DataFrame({
    'Number of Clusters (k)': k_values,
    'WCSS (Inertia)': [round(w, 2) for w in wcss_vals],
    'Silhouette Score': [round(s, 4) for s in sil_vals]
})
print(table1_df.to_string(index=False))


## 4. Model B: DBSCAN Clustering
DBSCAN discovers arbitrarily shaped clusters and flags low-density outliers as noise.


In [ ]:
nn = NearestNeighbors(n_neighbors=10)
nn.fit(X_pca50)
distances, _ = nn.kneighbors(X_pca50)
k_dist = np.sort(distances[:, -1])

eps_val = 14.5
min_pts = 20
dbscan = DBSCAN(eps=eps_val, min_samples=min_pts)
db_labels = dbscan.fit_predict(X_pca50)

n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_db_noise = np.sum(db_labels == -1)
print('DBSCAN: Formed', n_db_clusters, 'clusters with', n_db_noise, 'noise points.')


## 5. Model C: Hierarchical Agglomerative Clustering (HAC)
We evaluate bottom-up hierarchical clustering under Ward, Complete, Average, and Single linkage criteria.


In [ ]:
sub_idx = np.random.RandomState(SEED).choice(len(X_scaled), size=2500, replace=False)
X_sub = X_pca50[sub_idx]
y_sub = y_true[sub_idx]

linkage_types = ['ward', 'complete', 'average', 'single']
for link_type in linkage_types:
    if link_type == 'ward':
        Z = linkage(X_sub, method='ward')
    else:
        Z = linkage(X_sub, method=link_type, metric='euclidean')
    c_coeff, _ = cophenet(Z, pdist(X_sub))
    hac = AgglomerativeClustering(n_clusters=6, linkage=link_type)
    h_labels = hac.fit_predict(X_sub)
    ari = adjusted_rand_score(y_sub, h_labels)
    nmi = normalized_mutual_info_score(y_sub, h_labels)
    print('Linkage:', link_type, '| Cophenetic:', round(c_coeff, 4), '| ARI:', round(ari, 4), '| NMI:', round(nmi, 4))


## 6. Comparative Evaluation and Results
Comprehensive comparison across internal metrics (Silhouette, Davies-Bouldin, Calinski-Harabasz) and external metrics (ARI, NMI, Homogeneity, Completeness, V-measure).


In [ ]:
km_labels_6 = km_dict[6][1]
hac_ward_full = AgglomerativeClustering(n_clusters=6, linkage='ward')
hac_ward_labels = hac_ward_full.fit_predict(X_pca50)

print('K-Means (k=6) ARI:', round(adjusted_rand_score(y_true, km_labels_6), 4))
print('HAC (Ward, k=6) ARI:', round(adjusted_rand_score(y_true, hac_ward_labels), 4))
print('DBSCAN ARI:', round(adjusted_rand_score(y_true, db_labels), 4))
